<a href="https://colab.research.google.com/github/dhfricard-coder/2_multi-factor-equity-ranking/blob/main/01_data_acquisition.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
# DEFINING UNIVERSE (EQUITIES TO ANALYSE)
# Example using the sp500 historical constituents repo
import pandas as pd

#Use this link to get most up to date sp500 data: https://github.com/fja05680/sp500
url = "https://raw.githubusercontent.com/fja05680/sp500/refs/heads/master/S%26P%20500%20Historical%20Components%20%26%20Changes(01-17-2026).csv"
sp500_history = pd.read_csv(url)

In [4]:
## ENVIRONMENT SET UP

In [ ]:
# Install required packages
!pip install yfinance simfin python-dotenv -q

# Standard imports
import os
import time
import warnings
import numpy as np
import pandas as pd
import yfinance as yf
import simfin as sf
from simfin.names import *
from datetime import datetime, timedelta
from pathlib import Path

warnings.filterwarnings('ignore')
pd.set_option('display.max_columns', 50)

print("Environment ready.")

In [ ]:
## MOUNT GOOGLE DRIVE AND DEFINE PATHS

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

# Define project paths — adjust the project root to your Drive structure
PROJECT_ROOT = Path('/content/drive/MyDrive/multifactor-equity-engine')
DATA_RAW = PROJECT_ROOT / 'data' / 'raw'
DATA_PROCESSED = PROJECT_ROOT / 'data' / 'processed'
SIMFIN_DIR = DATA_RAW / 'simfin'

# Create directories if they don't exist
for path in [DATA_RAW, DATA_PROCESSED, SIMFIN_DIR]:
    path.mkdir(parents=True, exist_ok=True)

print(f"Project root: {PROJECT_ROOT}")
print(f"Directories ready.")

In [ ]:
##API KEY MANAGEMENT

In [ ]:
from dotenv import load_dotenv

# Load .env file from project root
env_path = PROJECT_ROOT / '.env'

if not env_path.exists():
    print(f"WARNING: No .env file at {env_path}")
    print("Create one with: SIMFIN_API_KEY=your_key_here")
else:
    load_dotenv(env_path)

SIMFIN_API_KEY = os.environ.get('SIMFIN_API_KEY', 'free')  # 'free' works for limited access

# Configure simfin
sf.set_api_key(SIMFIN_API_KEY)
sf.set_data_dir(str(SIMFIN_DIR))

print(f"Simfin configured. Data dir: {SIMFIN_DIR}")

In [ ]:
##DEFINE BACKTEST PARAMETERS

In [ ]:
# Backtest universe and date range
START_DATE = '2010-01-01'
END_DATE = '2024-12-31'

# Fundamental data lag (days after fiscal period end before data is "available")
# 90 days is the standard assumption for quarterly filings (10-Q deadline is 40-45 days,
# but we use 90 for safety to also cover annual 10-K filings which have 60-90 day deadlines)
FUNDAMENTAL_LAG_DAYS = 90

# Universe size — start with S&P 500 historical members
MIN_DOLLAR_VOLUME = 5_000_000  # $5M daily liquidity filter

print(f"Backtest period: {START_DATE} to {END_DATE}")
print(f"Fundamental availability lag: {FUNDAMENTAL_LAG_DAYS} days")

In [ ]:
##BUILD HISTORICAL UNIVERSE (S&P 500 COSNTITUENTS)

In [ ]:
def fetch_sp500_history():
    """
    Pull historical S&P 500 constituent membership.
    Returns a DataFrame with columns: date, tickers (list)

    Source: fja05680/sp500 GitHub repo, which maintains historical membership.
    """
    url = ("https://raw.githubusercontent.com/fja05680/sp500/master/"
           "S%26P%20500%20Historical%20Components%20%26%20Changes(08-17-2024).csv")

    df = pd.read_csv(url)
    df['date'] = pd.to_datetime(df['date'])
    df['tickers'] = df['tickers'].str.split(',')

    return df

sp500_history = fetch_sp500_history()
print(f"Loaded {len(sp500_history)} historical membership snapshots")
print(f"Date range: {sp500_history['date'].min()} to {sp500_history['date'].max()}")
print(f"\nSample of recent membership:")
print(sp500_history.tail(3))

In [ ]:
def get_universe_at_date(sp500_history, target_date):
    """
    Return the S&P 500 constituent list active at target_date.
    Uses the most recent membership snapshot at or before target_date.
    """
    target_date = pd.Timestamp(target_date)
    valid_snapshots = sp500_history[sp500_history['date'] <= target_date]

    if valid_snapshots.empty:
        return []

    most_recent = valid_snapshots.iloc[-1]
    return most_recent['tickers']

# Test: what was the S&P 500 on 2015-06-30?
test_universe = get_universe_at_date(sp500_history, '2015-06-30')
print(f"S&P 500 size on 2015-06-30: {len(test_universe)}")
print(f"Sample: {test_universe[:10]}")

In [ ]:
# Build the full set of tickers that have ever been in the S&P 500 during our backtest period
all_historical_tickers = set()
for tickers in sp500_history[
    (sp500_history['date'] >= START_DATE) &
    (sp500_history['date'] <= END_DATE)
]['tickers']:
    all_historical_tickers.update(tickers)

# Clean tickers: yfinance uses '-' for share classes (BRK-B not BRK.B)
all_historical_tickers = {t.strip().replace('.', '-') for t in all_historical_tickers if t.strip()}
all_historical_tickers = sorted(all_historical_tickers)

print(f"Total unique tickers across backtest period: {len(all_historical_tickers)}")
print(f"This is your full data download universe.")

In [ ]:
##DOWNLOAD PRICE DTA VIA YFINANCE

In [ ]:
def download_prices_batched(tickers, start_date, end_date, batch_size=50, sleep_seconds=2):
    """
    Download adjusted close prices for a list of tickers in batches.
    Returns a DataFrame indexed by date with one column per ticker.
    """
    all_prices = []
    failed_tickers = []

    n_batches = (len(tickers) + batch_size - 1) // batch_size

    for i in range(0, len(tickers), batch_size):
        batch = tickers[i:i+batch_size]
        batch_num = i // batch_size + 1
        print(f"Batch {batch_num}/{n_batches}: downloading {len(batch)} tickers...")

        try:
            data = yf.download(
                batch,
                start=start_date,
                end=end_date,
                auto_adjust=True,
                progress=False,
                threads=True
            )

            # Handle single vs multi-ticker response shape
            if len(batch) == 1:
                close = data[['Close']].rename(columns={'Close': batch[0]})
            else:
                close = data['Close']

            all_prices.append(close)

        except Exception as e:
            print(f"  Batch failed: {e}")
            failed_tickers.extend(batch)

        time.sleep(sleep_seconds)

    if not all_prices:
        return pd.DataFrame(), failed_tickers

    prices = pd.concat(all_prices, axis=1)
    prices = prices.loc[:, ~prices.columns.duplicated()]  # Remove any dupes

    return prices, failed_tickers

# Run the download — this takes ~10-15 minutes for ~700 tickers
prices_path = DATA_PROCESSED / 'prices_daily.parquet'

if prices_path.exists():
    print(f"Loading cached prices from {prices_path}")
    prices = pd.read_parquet(prices_path)
else:
    print(f"Downloading {len(all_historical_tickers)} tickers...")
    prices, failed = download_prices_batched(
        all_historical_tickers,
        START_DATE,
        END_DATE
    )
    prices.to_parquet(prices_path)
    print(f"Saved to {prices_path}")
    print(f"Failed tickers: {failed}")

print(f"\nPrices shape: {prices.shape}")
print(f"Date range: {prices.index.min()} to {prices.index.max()}")
print(f"Tickers: {prices.shape[1]}")

In [ ]:
##COMPUTE DAILY AND MONTHLY RETURNS

In [ ]:
# Daily returns
returns_daily = prices.pct_change()

# Monthly returns — use last business day of month
prices_monthly = prices.resample('M').last()
returns_monthly = prices_monthly.pct_change()

# Save
returns_daily.to_parquet(DATA_PROCESSED / 'returns_daily.parquet')
returns_monthly.to_parquet(DATA_PROCESSED / 'returns_monthly.parquet')
prices_monthly.to_parquet(DATA_PROCESSED / 'prices_monthly.parquet')

print(f"Daily returns shape: {returns_daily.shape}")
print(f"Monthly returns shape: {returns_monthly.shape}")
print(f"\nSample monthly returns (last 3 months, first 5 tickers):")
print(returns_monthly.iloc[-3:, :5])

In [ ]:
## COMPUTE DOLLAR VOLUME FOR LIQUIDITY FILTER

In [ ]:
def fetch_volume_data(tickers, start_date, end_date, batch_size=50, sleep_seconds=2):
    """Download daily volume for the same universe."""
    all_volumes = []

    for i in range(0, len(tickers), batch_size):
        batch = tickers[i:i+batch_size]
        try:
            data = yf.download(batch, start=start_date, end=end_date,
                             auto_adjust=False, progress=False, threads=True)
            volume = data['Volume'] if len(batch) > 1 else data[['Volume']].rename(
                columns={'Volume': batch[0]})
            all_volumes.append(volume)
        except Exception as e:
            print(f"  Volume batch failed: {e}")
        time.sleep(sleep_seconds)

    return pd.concat(all_volumes, axis=1) if all_volumes else pd.DataFrame()

volume_path = DATA_PROCESSED / 'volume_daily.parquet'

if volume_path.exists():
    volume = pd.read_parquet(volume_path)
else:
    volume = fetch_volume_data(all_historical_tickers, START_DATE, END_DATE)
    volume.to_parquet(volume_path)

# Dollar volume = price × volume, then 60-day rolling average
dollar_volume_daily = prices * volume
dollar_volume_60d = dollar_volume_daily.rolling(60).mean()

# Resample to month-end for use at rebalance dates
dollar_volume_monthly = dollar_volume_60d.resample('M').last()
dollar_volume_monthly.to_parquet(DATA_PROCESSED / 'dollar_volume_monthly.parquet')

print(f"Dollar volume (monthly) shape: {dollar_volume_monthly.shape}")
print(f"Sample (last month, top 5 by liquidity):")
print(dollar_volume_monthly.iloc[-1].sort_values(ascending=False).head())

In [ ]:
## DOWNLOAD FUNDAMENTAL DATA VIA SIMFIN

In [ ]:
def load_simfin_fundamentals():
    """
    Download all US fundamentals from Simfin.
    Returns three DataFrames: income, balance, cashflow.
    """
    print("Loading income statements...")
    df_income = sf.load_income(variant='quarterly', market='us')

    print("Loading balance sheets...")
    df_balance = sf.load_balance(variant='quarterly', market='us')

    print("Loading cash flow statements...")
    df_cashflow = sf.load_cashflow(variant='quarterly', market='us')

    return df_income, df_balance, df_cashflow

df_income, df_balance, df_cashflow = load_simfin_fundamentals()

print(f"\nIncome shape: {df_income.shape}")
print(f"Balance shape: {df_balance.shape}")
print(f"Cashflow shape: {df_cashflow.shape}")
print(f"\nIncome columns sample: {df_income.columns.tolist()[:10]}")

In [ ]:
## BUILD POINT-IN-TIME FUNDAMENTALS PANEL

In [ ]:
def build_pit_fundamentals(df_income, df_balance, df_cashflow, lag_days=90):
    """
    Build a point-in-time-correct fundamentals panel.

    Each row represents a (ticker, fiscal_period) observation, with an
    'available_date' column indicating when the data became publicly known.

    For a factor signal at date t, only use rows where available_date <= t.
    """
    # Reset indices to get Ticker and Report Date as columns
    inc = df_income.reset_index()
    bal = df_balance.reset_index()
    cf = df_cashflow.reset_index()

    # Standard merge keys
    merge_keys = ['Ticker', 'Report Date', 'Fiscal Year', 'Fiscal Period']

    # Merge income + balance + cashflow on ticker + fiscal period
    fundamentals = inc.merge(bal, on=merge_keys, how='outer', suffixes=('', '_bal'))
    fundamentals = fundamentals.merge(cf, on=merge_keys, how='outer', suffixes=('', '_cf'))

    # Construct available_date
    # Simfin has 'Publish Date' for many rows; if missing, fall back to Report Date + lag_days
    if 'Publish Date' in fundamentals.columns:
        fundamentals['Publish Date'] = pd.to_datetime(fundamentals['Publish Date'])
        fundamentals['Report Date'] = pd.to_datetime(fundamentals['Report Date'])

        fundamentals['available_date'] = fundamentals['Publish Date'].fillna(
            fundamentals['Report Date'] + pd.Timedelta(days=lag_days)
        )
    else:
        fundamentals['Report Date'] = pd.to_datetime(fundamentals['Report Date'])
        fundamentals['available_date'] = fundamentals['Report Date'] + pd.Timedelta(days=lag_days)

    # Sort for clarity
    fundamentals = fundamentals.sort_values(['Ticker', 'Report Date'])

    return fundamentals

fundamentals = build_pit_fundamentals(
    df_income, df_balance, df_cashflow,
    lag_days=FUNDAMENTAL_LAG_DAYS
)

# Save
fundamentals.to_parquet(DATA_PROCESSED / 'fundamentals_pit.parquet')

print(f"Point-in-time fundamentals shape: {fundamentals.shape}")
print(f"Unique tickers: {fundamentals['Ticker'].nunique()}")
print(f"\nSample row showing the PIT logic:")
print(fundamentals[['Ticker', 'Report Date', 'available_date']].head())

In [ ]:
## COMPUTE TRAILING TWELVE MONTH (TTM) AGGREGATES

In [ ]:
def compute_ttm_values(fundamentals, value_columns):
    """
    For each (ticker, fiscal period), compute trailing 4-quarter sums for flow variables.
    Stock variables (balance sheet items) are taken at the most recent quarter.

    value_columns: dict mapping column name -> 'flow' or 'stock'
        flow = sum last 4 quarters (e.g. revenue, net income)
        stock = use most recent (e.g. total assets, total equity)
    """
    df = fundamentals.copy()
    df = df.sort_values(['Ticker', 'Report Date'])

    for col, var_type in value_columns.items():
        if col not in df.columns:
            print(f"Warning: {col} not in fundamentals, skipping")
            continue

        if var_type == 'flow':
            # 4-quarter rolling sum within each ticker
            df[f'{col}_TTM'] = df.groupby('Ticker')[col].transform(
                lambda x: x.rolling(4, min_periods=4).sum()
            )
        elif var_type == 'stock':
            # Just use as-is (point-in-time balance)
            df[f'{col}_TTM'] = df[col]

    return df

# Define which variables are flows (income statement, cash flow) vs stocks (balance sheet)
# Adjust column names to match your simfin output — print fundamentals.columns to verify
ttm_spec = {
    'Revenue': 'flow',
    'Net Income': 'flow',
    'Operating Income (Loss)': 'flow',
    'Gross Profit': 'flow',
    'Net Cash from Operating Activities': 'flow',
    'Capital Expenditures': 'flow',
    'Total Assets': 'stock',
    'Total Equity': 'stock',
    'Total Liabilities': 'stock',
    'Long Term Debt': 'stock',
    'Cash, Cash Equivalents & Short Term Investments': 'stock',
    'Shares (Diluted)': 'stock',
}

fundamentals_ttm = compute_ttm_values(fundamentals, ttm_spec)
fundamentals_ttm.to_parquet(DATA_PROCESSED / 'fundamentals_ttm.parquet')

print(f"TTM fundamentals shape: {fundamentals_ttm.shape}")
print(f"\nSample TTM columns added: {[c for c in fundamentals_ttm.columns if 'TTM' in c]}")

In [ ]:
## BUILD AS-OF LOOKUP FUNCTION

In [ ]:
def get_fundamentals_as_of(fundamentals_ttm, ticker, as_of_date):
    """
    Return the most recent fundamental row available for `ticker` as of `as_of_date`.

    Returns None if no data was available by that date.
    """
    as_of_date = pd.Timestamp(as_of_date)

    ticker_data = fundamentals_ttm[fundamentals_ttm['Ticker'] == ticker]
    available = ticker_data[ticker_data['available_date'] <= as_of_date]

    if available.empty:
        return None

    return available.iloc[-1]


def build_fundamentals_panel(fundamentals_ttm, rebalance_dates, universe_func):
    """
    For each rebalance date, build a cross-sectional panel of available fundamentals.

    universe_func: callable that returns the ticker list for a given date

    Returns a long-format DataFrame: (date, ticker, ...all TTM columns)
    """
    panels = []

    for date in rebalance_dates:
        tickers_at_date = universe_func(date)
        rows = []

        for ticker in tickers_at_date:
            row = get_fundamentals_as_of(fundamentals_ttm, ticker, date)
            if row is not None:
                row_dict = row.to_dict()
                row_dict['rebalance_date'] = date
                rows.append(row_dict)

        if rows:
            panels.append(pd.DataFrame(rows))

    return pd.concat(panels, ignore_index=True) if panels else pd.DataFrame()


# Build monthly rebalance dates
rebalance_dates = pd.date_range(START_DATE, END_DATE, freq='M')

# Universe lookup function (clean ticker format for matching)
def universe_at(date):
    raw = get_universe_at_date(sp500_history, date)
    return [t.strip().replace('.', '-') for t in raw if t.strip()]

print(f"Building fundamentals panel across {len(rebalance_dates)} rebalance dates...")
print(f"This will take a few minutes...")

# Build a sample first to verify, then full panel
sample_panel = build_fundamentals_panel(
    fundamentals_ttm,
    rebalance_dates[-3:],  # last 3 months as test
    universe_at
)

print(f"\nSample panel shape: {sample_panel.shape}")
print(f"Sample columns: {sample_panel.columns.tolist()[:15]}")

In [ ]:
# Now build the full panel — this is the master fundamentals data structure
panel_path = DATA_PROCESSED / 'fundamentals_panel.parquet'

if panel_path.exists():
    fundamentals_panel = pd.read_parquet(panel_path)
    print(f"Loaded cached panel: {fundamentals_panel.shape}")
else:
    fundamentals_panel = build_fundamentals_panel(
        fundamentals_ttm, rebalance_dates, universe_at
    )
    fundamentals_panel.to_parquet(panel_path)
    print(f"Built and saved panel: {fundamentals_panel.shape}")

print(f"\nUnique rebalance dates: {fundamentals_panel['rebalance_date'].nunique()}")
print(f"Unique tickers: {fundamentals_panel['Ticker'].nunique()}")
print(f"Avg names per rebalance: {fundamentals_panel.groupby('rebalance_date').size().mean():.0f}")

In [ ]:
## FETCH SECTOR CLASSIFICATIONS

In [ ]:
def fetch_sectors(tickers, sleep_seconds=0.5):
    """
    Fetch GICS sector for each ticker via yfinance.
    Note: this returns CURRENT sector — historical sector reclassifications are not handled.
    Document this limitation in your README.
    """
    sectors = {}

    for i, ticker in enumerate(tickers):
        if i % 50 == 0:
            print(f"  {i}/{len(tickers)}...")

        try:
            info = yf.Ticker(ticker).info
            sectors[ticker] = info.get('sector', 'Unknown')
        except Exception as e:
            sectors[ticker] = 'Unknown'

        time.sleep(sleep_seconds)

    return pd.Series(sectors, name='sector')

sectors_path = DATA_PROCESSED / 'sectors.parquet'

if sectors_path.exists():
    sectors = pd.read_parquet(sectors_path)['sector']
    print(f"Loaded cached sectors: {len(sectors)} tickers")
else:
    print(f"Fetching sectors for {len(all_historical_tickers)} tickers...")
    sectors = fetch_sectors(all_historical_tickers)
    sectors.to_frame().to_parquet(sectors_path)

print(f"\nSector distribution:")
print(sectors.value_counts())

In [ ]:
## FINAL DATA VALIDATION

In [ ]:
def validate_data():
    """Run sanity checks on the assembled dataset."""
    issues = []

    # Check 1: prices and returns aligned
    if not prices_monthly.index.equals(returns_monthly.index):
        issues.append("Monthly prices and returns indices misaligned")

    # Check 2: No future-dated fundamentals
    max_pit = fundamentals_panel['rebalance_date'].max()
    panel_check = fundamentals_panel[
        fundamentals_panel['available_date'] > fundamentals_panel['rebalance_date']
    ]
    if len(panel_check) > 0:
        issues.append(f"{len(panel_check)} rows with future-dated fundamentals (LOOK-AHEAD BIAS)")

    # Check 3: Universe coverage at each date
    coverage = fundamentals_panel.groupby('rebalance_date').size()
    low_coverage = coverage[coverage < 100]
    if len(low_coverage) > 0:
        issues.append(f"{len(low_coverage)} rebalance dates with <100 names available")

    # Check 4: Sector coverage
    panel_tickers = set(fundamentals_panel['Ticker'].unique())
    sector_tickers = set(sectors.index)
    missing_sectors = panel_tickers - sector_tickers
    if missing_sectors:
        issues.append(f"{len(missing_sectors)} tickers missing sector classification")

    # Check 5: Price coverage
    missing_prices = panel_tickers - set(prices_monthly.columns)
    if missing_prices:
        issues.append(f"{len(missing_prices)} tickers in fundamentals but missing prices")

    return issues

issues = validate_data()

if issues:
    print("DATA VALIDATION ISSUES:")
    for issue in issues:
        print(f"  - {issue}")
else:
    print("All validation checks passed.")

print(f"\n=== Phase 2 Complete ===")
print(f"Files saved in {DATA_PROCESSED}:")
for f in sorted(DATA_PROCESSED.glob('*.parquet')):
    size_mb = f.stat().st_size / 1024 / 1024
    print(f"  {f.name}: {size_mb:.1f} MB")

In [ ]:
## QUICK VISUALISATION SANITY CHECK

In [ ]:
import matplotlib.pyplot as plt

fig, axes = plt.subplots(2, 2, figsize=(14, 8))

# 1. Universe size over time
universe_size = fundamentals_panel.groupby('rebalance_date').size()
axes[0, 0].plot(universe_size.index, universe_size.values)
axes[0, 0].set_title('Available Universe Size Over Time')
axes[0, 0].set_ylabel('Number of stocks')
axes[0, 0].axhline(500, color='red', linestyle='--', alpha=0.5, label='S&P 500 target')
axes[0, 0].legend()

# 2. SPY-like benchmark (equal weight) cumulative return
eq_weight_return = returns_monthly.mean(axis=1)
cum_return = (1 + eq_weight_return.fillna(0)).cumprod()
axes[0, 1].plot(cum_return.index, cum_return.values)
axes[0, 1].set_title('Equal-Weight Universe Cumulative Return')
axes[0, 1].set_ylabel('Growth of $1')

# 3. Sector distribution
sectors.value_counts().plot(kind='barh', ax=axes[1, 0])
axes[1, 0].set_title('Sector Distribution (Current GICS)')

# 4. Fundamentals coverage heatmap (sample)
key_cols = ['Revenue_TTM', 'Net Income_TTM', 'Total Assets_TTM']
coverage_pct = fundamentals_panel.groupby('rebalance_date')[key_cols].apply(
    lambda x: x.notna().mean()
)
coverage_pct.plot(ax=axes[1, 1])
axes[1, 1].set_title('Fundamentals Coverage by Date')
axes[1, 1].set_ylabel('% of universe with data')
axes[1, 1].legend(loc='lower right', fontsize=8)

plt.tight_layout()
plt.savefig(PROJECT_ROOT / 'outputs' / 'figures' / 'phase2_data_validation.png', dpi=100)
plt.show()